# Song Popularity EDA

**Student:** Bibek Shah | **ID:** 23189619

Exploratory Data Analysis of the Song Popularity Dataset with visualisations for the MLOps project report.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

df = pd.read_csv('song_popularity.csv')
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.info()

### 1. Target Variable Distribution

**RQ: What is the distribution of song popularity scores?**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['song_popularity'], bins=30, color='royalblue', edgecolor='white')
axes[0].set_title('Distribution of Song Popularity')
axes[0].set_xlabel('Popularity Score')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df['song_popularity'].mean(), color='red', ls='--', label=f"Mean: {df['song_popularity'].mean():.1f}")
axes[0].legend()

df['popularity_level'] = pd.cut(df['song_popularity'],
                                bins=[-1, 25, 50, 75, 101],
                                labels=['Low (0-25)', 'Medium (26-50)', 'High (51-75)', 'Very High (76-100)'])
counts = df['popularity_level'].value_counts()
colors = ['#ff6b6b', '#ffd93d', '#6bcb77', '#4d96ff']
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, wedgeprops={'edgecolor': 'white'})
axes[1].set_title('Popularity Level Distribution')

plt.tight_layout()
plt.savefig('popularity_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 2. Correlation Heatmap

**RQ: Which audio features correlate most with song popularity?**

In [ ]:
feature_cols = [c for c in df.columns if c not in ['song_name', 'popularity_level']]
corr = df[feature_cols].corr()

mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5, square=True,
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top features correlated with popularity
popularity_corr = corr['song_popularity'].drop('song_popularity').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(popularity_corr.index, popularity_corr.values,
               color=['#4d96ff' if v > 0 else '#ff6b6b' for v in popularity_corr.values])
ax.axvline(0, color='black', linewidth=0.5)
ax.set_title('Feature Correlation with Song Popularity', fontsize=14, fontweight='bold')
ax.set_xlabel('Pearson Correlation Coefficient')
for bar, val in zip(bars, popularity_corr.values):
    ax.text(val + (0.01 if val > 0 else -0.04), bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10)
plt.tight_layout()
plt.savefig('popularity_correlation_bars.png', dpi=150, bbox_inches='tight')
plt.show()

### 3. Song Popularity by Energy Level (RQ1)

**RQ1: How does song popularity vary across different energy levels?**

In [ ]:
df['energy_level'] = pd.cut(df['energy'], bins=[-0.01, 0.33, 0.66, 1.01],
                            labels=['Low Energy', 'Medium Energy', 'High Energy'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x='energy_level', y='song_popularity',
            palette='Set2', ax=axes[0])
axes[0].set_title('Popularity by Energy Level', fontweight='bold')
axes[0].set_xlabel('Energy Level')
axes[0].set_ylabel('Popularity Score')

energy_means = df.groupby('energy_level', observed=True)['song_popularity'].mean()
axes[1].bar(energy_means.index, energy_means.values, color=['#ff6b6b', '#ffd93d', '#6bcb77'],
            edgecolor='white', linewidth=1.5)
axes[1].set_title('Mean Popularity by Energy Level', fontweight='bold')
axes[1].set_xlabel('Energy Level')
axes[1].set_ylabel('Mean Popularity')
for i, v in enumerate(energy_means.values):
    axes[1].text(i, v + 0.5, f'{v:.1f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('popularity_by_energy.png', dpi=150, bbox_inches='tight')
plt.show()
print(energy_means)

### 4. Danceability by Popularity Level (RQ2)

**RQ2: Do more popular songs have higher danceability?**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x='popularity_level', y='danceability',
            palette='Set2', ax=axes[0])
axes[0].set_title('Danceability by Popularity Level', fontweight='bold')
axes[0].set_xlabel('Popularity Level')
axes[0].set_ylabel('Danceability')

sns.violinplot(data=df, x='popularity_level', y='danceability',
               palette='Set2', ax=axes[1])
axes[1].set_title('Danceability Distribution by Popularity', fontweight='bold')
axes[1].set_xlabel('Popularity Level')
axes[1].set_ylabel('Danceability')

plt.tight_layout()
plt.savefig('danceability_by_popularity.png', dpi=150, bbox_inches='tight')
plt.show()

print('Mean Danceability by Popularity Level:')
print(df.groupby('popularity_level', observed=True)['danceability'].mean().round(3))

### 5. Musical Key vs Popularity Level (RQ3)

**RQ3: Does musical key influence song popularity?**

In [ ]:
key_names = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
df['key_name'] = df['key'].map(lambda x: key_names[int(x)] if 0 <= int(x) <= 11 else 'Unknown')

key_pop = df.groupby('key_name')['song_popularity'].mean().reindex(key_names)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(key_pop.index, key_pop.values, color='royalblue', edgecolor='white')
ax.set_title('Mean Song Popularity by Musical Key', fontweight='bold')
ax.set_xlabel('Musical Key')
ax.set_ylabel('Mean Popularity Score')
ax.axhline(df['song_popularity'].mean(), color='red', ls='--',
           label=f'Overall Mean: {df["song_popularity"].mean():.1f}')
ax.legend()

for bar, val in zip(bars, key_pop.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('popularity_by_key.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
ct = pd.crosstab(df['key_name'], df['popularity_level'])
fig, ax = plt.subplots(figsize=(14, 6))
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
bottom = np.zeros(len(key_names))
colors = ['#ff6b6b', '#ffd93d', '#6bcb77', '#4d96ff']
for idx, level in enumerate(ct_pct.columns):
    ax.bar(key_names, ct_pct[level], bottom=bottom, label=level, color=colors[idx], edgecolor='white')
    bottom += ct_pct[level]
ax.set_title('Musical Key Distribution Across Popularity Levels', fontweight='bold')
ax.set_xlabel('Musical Key')
ax.set_ylabel('Percentage (%)')
ax.legend(title='Popularity Level')
plt.tight_layout()
plt.savefig('key_popularity_stacked.png', dpi=150, bbox_inches='tight')
plt.show()

### 6. Feature Distributions (Histograms)

In [ ]:
features = ['acousticness', 'danceability', 'energy', 'instrumentalness',
            'liveness', 'loudness', 'speechiness', 'tempo', 'audio_valence', 'song_duration_ms']

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, feat in enumerate(features):
    axes[i].hist(df[feat], bins=40, color='royalblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(feat.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    axes[i].axvline(df[feat].mean(), color='red', ls='--', lw=1.5)

plt.suptitle('Feature Distributions (Red = Mean)', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

### 7. Audio Mode (Major/Minor) vs Popularity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

mode_labels = ['Minor', 'Major']
mode_data = df.groupby('audio_mode')['song_popularity']

axes[0].bar(mode_labels, mode_data.mean().values,
            color=['#ff6b6b', '#4d96ff'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Mean Popularity: Minor vs Major', fontweight='bold')
axes[0].set_ylabel('Mean Popularity')
for i, v in enumerate(mode_data.mean().values):
    axes[0].text(i, v + 0.3, f'{v:.1f}', ha='center', fontweight='bold')

mode_counts = df['audio_mode'].map({0: 'Minor', 1: 'Major'}).value_counts()
axes[1].pie(mode_counts.values, labels=mode_counts.index, autopct='%1.1f%%',
            colors=['#ff6b6b', '#4d96ff'], startangle=90, wedgeprops={'edgecolor': 'white'})
axes[1].set_title('Songs in Minor vs Major Key')

plt.tight_layout()
plt.savefig('mode_popularity.png', dpi=150, bbox_inches='tight')
plt.show()

### 8. Time Signature vs Popularity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ts_counts = df['time_signature'].value_counts().sort_index()
axes[0].bar(ts_counts.index.astype(str), ts_counts.values,
            color='royalblue', edgecolor='white')
axes[0].set_title('Number of Songs by Time Signature', fontweight='bold')
axes[0].set_xlabel('Time Signature')
axes[0].set_ylabel('Count')

ts_means = df.groupby('time_signature')['song_popularity'].mean()
axes[1].bar(ts_means.index.astype(str), ts_means.values,
            color='#6bcb77', edgecolor='white')
axes[1].set_title('Mean Popularity by Time Signature', fontweight='bold')
axes[1].set_xlabel('Time Signature')
axes[1].set_ylabel('Mean Popularity')
for i, v in enumerate(ts_means.values):
    axes[1].text(i, v + 0.3, f'{v:.1f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('time_signature_popularity.png', dpi=150, bbox_inches='tight')
plt.show()

### 9. Pairplot of Key Features

In [ ]:
# Sample 500 rows for faster plotting
sample = df.sample(500, random_state=42)
pair_features = ['song_popularity', 'danceability', 'energy', 'loudness', 'acousticness', 'audio_valence']

g = sns.pairplot(sample[pair_features], diag_kind='kde', corner=True,
                 plot_kws={'alpha': 0.5, 's': 15, 'color': 'royalblue'})
g.fig.suptitle('Pairplot of Key Audio Features', fontsize=14, fontweight='bold', y=1.02)
plt.savefig('pairplot_features.png', dpi=150, bbox_inches='tight')
plt.show()

### 10. Top 10 Most & Least Popular Songs

In [ ]:
top10 = df.nlargest(10, 'song_popularity')[['song_name', 'song_popularity']]
bottom10 = df.nsmallest(10, 'song_popularity')[['song_name', 'song_popularity']]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh(range(10), top10['song_popularity'].values, color='#6bcb77', edgecolor='white')
axes[0].set_yticks(range(10))
axes[0].set_yticklabels(top10['song_name'].values, fontsize=9)
axes[0].invert_yaxis()
axes[0].set_title('Top 10 Most Popular Songs', fontweight='bold')
axes[0].set_xlabel('Popularity Score')

axes[1].barh(range(10), bottom10['song_popularity'].values, color='#ff6b6b', edgecolor='white')
axes[1].set_yticks(range(10))
axes[1].set_yticklabels(bottom10['song_name'].values, fontsize=9)
axes[1].invert_yaxis()
axes[1].set_title('Top 10 Least Popular Songs', fontweight='bold')
axes[1].set_xlabel('Popularity Score')

plt.tight_layout()
plt.savefig('top_bottom_songs.png', dpi=150, bbox_inches='tight')
plt.show()

### 11. Loudness vs Energy Scatter

In [ ]:
scatter_sample = df.sample(1000, random_state=42)

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(scatter_sample['energy'], scatter_sample['loudness'],
                     c=scatter_sample['song_popularity'], cmap='viridis',
                     alpha=0.6, s=30, edgecolors='white', linewidth=0.3)
ax.set_xlabel('Energy')
ax.set_ylabel('Loudness (dB)')
ax.set_title('Energy vs Loudness Colored by Popularity', fontsize=14, fontweight='bold')
cbar = plt.colorbar(scatter)
cbar.set_label('Popularity Score')
plt.tight_layout()
plt.savefig('energy_loudness_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

### Summary of Findings

- **Target distribution**: song_popularity is left-skewed — most songs cluster between 40-75
- **Top correlation**: `loudness` and `energy` have the strongest positive correlation with popularity
- **Energy**: Higher-energy songs tend to be more popular
- **Danceability**: More popular songs have slightly higher danceability
- **Mode**: Major key songs are more common and slightly more popular on average
- **Key**: Key of C has the highest mean popularity; keys D, G also perform well
- **Time signature**: 4/4 dominates overwhelmingly (~97% of songs)
- **Null values**: None — the dataset is complete